# Pancancer enrichment analysis step 12: GSEApy PreRank with 1 - p

## Setup

In [ ]:
import pandas as pd
import numpy as np
import os
import datetime
import gseapy as gp

In [ ]:
TIME_START = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
NUM_PERMUTATIONS = 1000

STEP1_DIR = "step1_outputs"
STEP1_FILE = "tumor_change_20200529_195104_10000_perm.tsv"
STEP1_FILE_PATH = os.path.join(STEP1_DIR, STEP1_FILE)

STEP12_DIR = "step12_outputs"

# Create log dir and file
LOG_DIR = "step12_checkpoints"
LOG_FILE = f"{TIME_START}_prerank.log"
LOG_FILE_PATH = os.path.join(LOG_DIR, LOG_FILE)

if not os.path.isdir(LOG_DIR):
    os.mkdir(LOG_DIR)
    
with open(LOG_FILE_PATH, 'w') as fp: 
    fp.write(f"{TIME_START}\n")
    fp.write(f"{datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - Started step 12 with {NUM_PERMUTATIONS} permutations.\n")

GSEAPY_DIR_PATH = os.path.join(STEP12_DIR, "gseapy")

if not os.path.isdir(STEP12_DIR):
    os.mkdir(STEP12_DIR)

## Prepare data

In [ ]:
# Read in the file from step 1
data = pd.read_csv(STEP1_FILE_PATH, sep='\t', index_col=0)
data = data[["cancer_type", "protein_str", "adj_p"]]

In [ ]:
# Create 1 - adj_p column
data = data.\
    assign(one_minus_adj_p=1 - data["adj_p"]).\
    loc[:, ["cancer_type", "protein_str", "one_minus_adj_p"]].\
    groupby(["cancer_type", "protein_str"]).\
    mean().\
    reset_index()

In [ ]:
data

## Run GSEApy PreRank analysis

In [ ]:
def gseapy_prerank_reactome(ranks, asc, wst):
    
    with open(LOG_FILE_PATH, 'a') as fp: 
        fp.write(f"\n{datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - Started asc={asc} wst={wst}.\n")
    
    output_file = f"enrichment_gseapy_prerank_1map_meandup_{TIME_START}_{NUM_PERMUTATIONS}_perms_asc_{asc}_wst_{wst}.tsv"
    output_path = os.path.join(STEP12_DIR, output_file)
    
    # For each cancer, find enriched pathways.
    all_enrichments = pd.DataFrame()

    for cancer_type in ranks["cancer_type"].unique():

        prot = ranks.loc[
            ranks["cancer_type"] == cancer_type,
            ["protein_str", "one_minus_adj_p"]
        ]
        
        pre_res = gp.prerank(
            rnk=prot,
            gene_sets="gene_set_libraries/Reactome_2020.gmt",
            outdir=GSEAPY_DIR_PATH,
            permutation_num=NUM_PERMUTATIONS,
            min_size=1,
            max_size=500, 
            weighted_score_type=wst,
            ascending=asc,
            no_plot=True,
            processes=1,
            seed=0)

        cancer_enriched = pre_res.res2d.assign(cancer_type=cancer_type)
        all_enrichments = all_enrichments.append(cancer_enriched)

        # Log that we finished this cancer type
        with open(LOG_FILE_PATH, 'a') as fp: 
            fp.write(f"{datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - Finished {cancer_type} data.\n")
            
    # Record the results
    all_enrichments.to_csv(output_path, sep="\t")
            
    with open(LOG_FILE_PATH, 'a') as fp: 
        fp.write(f"{datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - Finished asc={asc} wst={wst}.\n")

In [ ]:
gseapy_prerank_reactome(data, asc=False, wst=1)